# GOLD ATP PLAYER-TOURNAMENTS STATS

## Imports

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.window import Window
import pandas as pd

import os
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

from dotenv import load_dotenv
load_dotenv()

True

## Init spark

In [3]:
try:
    spark = SparkSession.builder.appName("fact_player_tournament_stats").getOrCreate()
except Exception as e:
    print(e)

In [4]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 200)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 50)

## Load database

In [5]:
# sv_atp_tournaments = (
#     spark.read
#     .format("jdbc")
#     .option("url", os.getenv("JDBC_URL"))
#     .option("dbtable", "silver.tb_atp_tournaments")
#     .option("user", os.getenv("DB_USER"))
#     .option("password", os.getenv("DB_PASSWORD"))
#     .option("driver", "org.postgresql.Driver")
#     .load()
#     )

tb_tournaments = spark.read.csv(r'C:\Users\phrug\OneDrive\Documentos\Repositórios\tennis-data-analytics\data\gold\dimension\dim_tournaments.csv', sep=',', header=True)
tb_entry = spark.read.csv(r'C:\Users\phrug\OneDrive\Documentos\Repositórios\tennis-data-analytics\data\gold\dimension\dim_entry.csv', sep=',', header=True)
tb_players = spark.read.csv(r'C:\Users\phrug\OneDrive\Documentos\Repositórios\tennis-data-analytics\data\gold\dimension\dim_players.csv', sep=',', header=True)
tb_player_match = spark.read.csv(r'C:\Users\phrug\OneDrive\Documentos\Repositórios\tennis-data-analytics\data\silver\tb_atp_player_match.csv', sep=',', header=True)
tb_matches = spark.read.csv(r'C:\Users\phrug\OneDrive\Documentos\Repositórios\tennis-data-analytics\data\silver\tb_atp_matches.csv', sep=',', header=True)

In [5]:
(
    tb_player_match
    .select(
        "MATCH_SCORE", # Não sei oq daria para fazer com isso, talvez placar de derrota, mas ai seria melhor ter o id da partida que fez ele perder, total de sets, games...
        "MATCH_ROUND", # Saber qual foi o round que ele acabou no torneio, sendo campeão ou não
        "MATCH_DURATION_M", # Saber mintos jogados no campeonato e/ou partida mais longa
        "PLAYER_RANK", # Saber quanto ele subiu entre um campeonato e outro (é o mesmo durante todo o campeonato)
        "PLAYER_RANK_PTS", # Saber quanto ele ganhou entre um campoenato e outro (é o mesmo durante todo o campeanato)
        "PLAYER_SEED",  # Saber se ele é cabeça de chave...
        "PLAYER_ACES", # Total de aces no campeanto e/ou media de aces por partida
        "PLAYER_DB_FAULTS", # Total de duplas faltas no campeanto e/ou media de duplas faltas por partida
        "PLAYER_SERVE_PTS", # Total de pontos de saque e/ou media de pontos de saque por partida
        "PLAYER_1ST_SERVES_IN", # Total de aces no campeanto e/ou media de aces por partida
        "PLAYER_1ST_SERVE_PTS_WON", # Total de primeiros serviços vencidos e/ou media de primeiros serviços vencidos por partida
        "PLAYER_2ND_SERVE_PTS_WON", # Total de segundos serviços vencidos e/ou media de segundos serviços vencidos por partida
        "PLAYER_SERVE_GAMES", # Total de games de saque e/ou media de games de saque por partida
        "PLAYER_BP_SAVED", # Total de break points salvos no campeanto e/ou media de break points salvos por partida
        "PLAYER_BP_FACED" # Total de break points sofridos no campeanto e/ou media de break points sofridos por partida
    )
    .limit(1)
)

MATCH_SCORE,MATCH_ROUND,MATCH_DURATION_M,PLAYER_RANK,PLAYER_RANK_PTS,PLAYER_SEED,PLAYER_ACES,PLAYER_DB_FAULTS,PLAYER_SERVE_PTS,PLAYER_1ST_SERVES_IN,PLAYER_1ST_SERVE_PTS_WON,PLAYER_2ND_SERVE_PTS_WON,PLAYER_SERVE_GAMES,PLAYER_BP_SAVED,PLAYER_BP_FACED
6-3 7-5,R64,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL


## Tournaments

In [6]:
tb_matches.limit(1)

MATCH_ID,TOURNEY_ID,MATCH_DRAW_SIZE,MATCH_DATE,MATCH_SCORE,MATCH_BEST_OF,MATCH_ROUND,MATCH_DURATION_M,PLAYER_W_ID,PLAYER_W_RANK,PLAYER_W_RANK_PTS,PLAYER_W_SEED,PLAYER_W_ENTRY,PLAYER_W_ACES,PLAYER_W_DB_FAULTS,PLAYER_W_SERVE_PTS,PLAYER_W_1ST_SERVES_IN,PLAYER_W_1ST_SERVE_PTS_WON,PLAYER_W_2ND_SERVE_PTS_WON,PLAYER_W_SERVE_GAMES,PLAYER_W_BP_SAVED,PLAYER_W_BP_FACED,PLAYER_L_ID,PLAYER_L_RANK,PLAYER_L_RANK_PTS,PLAYER_L_SEED,PLAYER_L_ENTRY,PLAYER_L_ACES,PLAYER_L_DB_FAULTS,PLAYER_L_SERVE_PTS,PLAYER_L_1ST_SERVES_IN,PLAYER_L_1ST_SERVE_PTS_WON,PLAYER_L_2ND_SERVE_PTS_WON,PLAYER_L_SERVE_GAMES,PLAYER_L_BP_SAVED,PLAYER_L_BP_FACED
1968-316-271,1968-316,32,19680708,3-6 6-1 6-2 1-6 6-2,5,R32,NULL,109821,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,117334,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL


In [7]:
from pyspark.sql import functions as f

fact_player_tournament = (
    tb_player_match
    .groupBy("TOURNEY_ID", "PLAYER_ID")
    .agg(
        # 1. Total de jogos disputados pelo atleta no torneio
        f.count("MATCH_ID").alias("TOTAL_MATCHES"),
        
        # 2. Total de vitórias no torneio
        f.sum(f.when(f.col("PLAYER_IS_WINNER") == True, 1).otherwise(0)).alias("TOTAL_WINS"),
        
        # 5. Flag: Foi o Campeão do torneio? (Ganhou a partida da Final)
        f.max(
            f.when((f.col("MATCH_ROUND") == "F") & (f.col("PLAYER_IS_WINNER") == True), 1).otherwise(0)
        ).alias("IS_CHAMPION"),
 
        # 6. Puxa a última rodada alcançada pelo jogador no torneio
        f.max_by("MATCH_ROUND", f.col("MATCH_NUM").cast('int')).alias("LAST_ROUND_PLAYED")
    )
)

In [7]:
fact_player_tournament.where("IS_CHAMPION = 1 AND LAST_ROUND_PLAYED <> 'F'").count()

NameError: name 'fact_player_tournament' is not defined

In [10]:
window = Window.partitionBy("PLAYER_ID", "TOURNEY_ID")

df = (
    tb_player_match.alias("m")
    .join(tb_tournaments.alias("t"), "TOURNEY_ID", 'left')
    .join(tb_players.alias("p"), "PLAYER_ID", 'left')

    
    .select(
        f.col("p.SK_PLAYER"),
        f.col("t.SK_TOURNEY"),
    )
    .distinct()
)

## Save dataframe

### Local

In [11]:
df.toPandas().to_csv(
    r"../../../data/gold/fact/fact_player_tournament_stats.csv",
    index=False,
    sep=",",
    encoding="utf-8"
)

Py4JJavaError: An error occurred while calling o167.collectToPython.
: java.lang.OutOfMemoryError: Java heap space
	at scala.collection.mutable.ResizableArray.ensureSize(ResizableArray.scala:106)
	at scala.collection.mutable.ResizableArray.ensureSize$(ResizableArray.scala:96)
	at scala.collection.mutable.ArrayBuffer.ensureSize(ArrayBuffer.scala:49)
	at scala.collection.mutable.ArrayBuffer.$plus$eq(ArrayBuffer.scala:85)
	at org.apache.spark.sql.execution.SparkPlan.$anonfun$executeCollect$2(SparkPlan.scala:449)
	at org.apache.spark.sql.execution.SparkPlan$$Lambda$4327/0x00000229d7065ec8.apply(Unknown Source)
	at scala.collection.Iterator.foreach(Iterator.scala:943)
	at scala.collection.Iterator.foreach$(Iterator.scala:943)
	at org.apache.spark.util.NextIterator.foreach(NextIterator.scala:21)
	at org.apache.spark.sql.execution.SparkPlan.$anonfun$executeCollect$1(SparkPlan.scala:449)
	at org.apache.spark.sql.execution.SparkPlan.$anonfun$executeCollect$1$adapted(SparkPlan.scala:4

### Supabase

In [ ]:
(
df.write
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "gold.fact_player_tournament_stats")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .mode("overwrite")
    .save()
)